In [ ]:
%pip install -r ../../requirements.txt

In [2]:
from pathlib import Path
import sys

import stim
import sinter
import numpy as np

# Decoder imports
import pymatching
import ldpc
from ldpc.sinter_decoders import SinterBpOsdDecoder, SinterLsdDecoder
notebook_dir = Path.cwd()

sys.path.append(str(notebook_dir.parent.parent / "decoders"))
from sinter_decoders.sinter_unionfind_decoder import SinterUnionFindDecoder

# Plotting
import matplotlib.pyplot as plt

In [ ]:
distances = [3, 5,7]
dep_error_rates = np.linspace(1e-4, 1e-2, 20)

# Source: adapted from https://github.com/quantumlib/Stim/blob/01f1aabd0fbc64d6943d42094a4bde96818399bd/doc/getting_started.ipynb

tasks = [
    sinter.Task(
        circuit=stim.Circuit.generated(
            "surface_code:rotated_memory_z",
            #"color_code:memory_xyz",
            rounds=d * 3,
            distance=d,
            before_round_data_depolarization=p, 
            after_clifford_depolarization=p, 
            before_measure_flip_probability=p, 
            after_reset_flip_probability=p
        ),
        json_metadata={'d': d, 'p': p},
    )
    for d in distances
    for p in dep_error_rates
]

In [ ]:
custom_decoders = {
    "bposd": SinterBpOsdDecoder(
        max_iter=10,
        bp_method="ms",
        ms_scaling_factor=0.625,
        schedule="parallel",
        osd_method="osd0",
        osd_order=0,
    ),

    # Union-find decoder for sinter, adapted from https://github.com/quantumgizmos/ldpc/blob/main/src_python/ldpc/sinter_decoders/sinter_bposd_decoder.py
    "union_find": SinterUnionFindDecoder(
        uf_method='False'
    ),
}

collected_stats = sinter.collect(
    num_workers=10,
    tasks=tasks,
    decoders=['pymatching','bposd','union_find'],
    custom_decoders=custom_decoders,
    #max_shots=1_000_000,
    max_shots=10_000,
    max_errors=1000,
)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(15,10))
sinter.plot_error_rate(
    ax=ax,
    stats=collected_stats,
    x_func=lambda stats: stats.json_metadata['p'],
    group_func=lambda stats: f"{stats.decoder},d={stats.json_metadata['d']}",
)
ax.axline((0, 0), slope=1, label='p', linestyle='--', color='black')
ax.loglog()
ax.set_title("Surface Code Error Rates")
ax.set_xlabel("Phyical Error Rate")
ax.set_ylabel("Logical Error Rate per Shot")
ax.grid(which='major')
ax.grid(which='minor')
ax.legend()

decoders = ["pymatching", "bposd", "union_find"]

decoder_times = []
decoder_labels = []

for decoder in decoders:
    total_seconds = 0
    total_shots = 0

    for stat in collected_stats:
        if stat.decoder == decoder:
            total_seconds += stat.seconds
            total_shots += stat.shots

    avg_microseconds_per_shot = 1e6 * total_seconds / total_shots

    decoder_labels.append(decoder)
    decoder_times.append(avg_microseconds_per_shot)

fig, ax = plt.subplots(1, 1, figsize=(7, 5))

ax.bar(decoder_labels, decoder_times)

ax.set_title("Average Decoder Runtime")
ax.set_xlabel("Decoder")
ax.set_ylabel("Runtime per Shot (µs)")
ax.grid(axis="y")
plt.show()

## Different code
Now we do the same thing but rather than a pair like code (surface code), we transition to a steane code, where we expect pymatching and union find to be be outperformed by bp

In [3]:
distances = [3, 5,7]
dep_error_rates = np.linspace(1e-4, 1e-2, 20)

# Source: adapted from https://github.com/quantumlib/Stim/blob/01f1aabd0fbc64d6943d42094a4bde96818399bd/doc/getting_started.ipynb

tasks = [
    sinter.Task(
        circuit=stim.Circuit.generated(
            "color_code:memory_xyz",
            rounds=d * 3,
            distance=d,
            before_round_data_depolarization=p, 
            after_clifford_depolarization=p, 
            before_measure_flip_probability=p, 
            after_reset_flip_probability=p
        ),
        json_metadata={'d': d, 'p': p},
    )
    for d in distances
    for p in dep_error_rates
]

In [ ]:
custom_decoders = {
    "bposd": SinterBpOsdDecoder(
        max_iter=10,
        bp_method="ms",
        ms_scaling_factor=0.625,
        schedule="parallel",
        osd_method="osd0",
        osd_order=0,
    ),

    "bplsd": SinterLsdDecoder(
        max_iter=10,
        bp_method="ms",
        ms_scaling_factor=0.625,
        schedule="parallel",
        omp_thread_count=4,
        serial_schedule_order=None,
        lsd_order=0,
    ),
}

collected_stats = sinter.collect(
    num_workers=10,
    tasks=tasks,
    decoders=['bposd','bplsd'],
    custom_decoders=custom_decoders,
    #max_shots=1_000_000,
    max_shots=10_000,
    max_errors=1000,
)

RuntimeError: Worker failed: traceback=Traceback (most recent call last):
  File "c:\Users\kjell\AppData\Local\Programs\Python\Python313\Lib\site-packages\sinter\_collection\_collection_worker_state.py", line 246, in run_message_loop
    did_some_work = self.do_some_work()
  File "c:\Users\kjell\AppData\Local\Programs\Python\Python313\Lib\site-packages\sinter\_collection\_collection_worker_state.py", line 217, in do_some_work
    some_work_done = self.compiled_sampler.sample(self.current_task_shots_left)
  File "c:\Users\kjell\AppData\Local\Programs\Python\Python313\Lib\site-packages\sinter\_collection\_sampler_ramp_throttled.py", line 51, in sample
    result = self.sub_sampler.sample(actual_shots)
  File "c:\Users\kjell\AppData\Local\Programs\Python\Python313\Lib\site-packages\sinter\_decoding\_stim_then_decode_sampler.py", line 193, in sample
    predictions = self.compiled_decoder.decode_shots_bit_packed(bit_packed_detection_event_data=dets)
  File "c:\Users\kjell\AppData\Local\Pro

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(15,10))
sinter.plot_error_rate(
    ax=ax,
    stats=collected_stats,
    x_func=lambda stats: stats.json_metadata['p'],
    group_func=lambda stats: f"{stats.decoder},d={stats.json_metadata['d']}",
)
ax.axline((0, 0), slope=1, label='p', linestyle='--', color='black')
ax.loglog()
ax.set_title("Surface Code Error Rates")
ax.set_xlabel("Phyical Error Rate")
ax.set_ylabel("Logical Error Rate per Shot")
ax.grid(which='major')
ax.grid(which='minor')
ax.legend()

decoders = ['bposd','bplsd']

decoder_times = []
decoder_labels = []

for decoder in decoders:
    total_seconds = 0
    total_shots = 0

    for stat in collected_stats:
        if stat.decoder == decoder:
            total_seconds += stat.seconds
            total_shots += stat.shots

    avg_microseconds_per_shot = 1e6 * total_seconds / total_shots

    decoder_labels.append(decoder)
    decoder_times.append(avg_microseconds_per_shot)

fig, ax = plt.subplots(1, 1, figsize=(7, 5))

ax.bar(decoder_labels, decoder_times)

ax.set_title("Average Decoder Runtime")
ax.set_xlabel("Decoder")
ax.set_ylabel("Runtime per Shot (µs)")
ax.grid(axis="y")
plt.show()